### **FINE-TUNING DATASET CREATION**

In [ ]:
# ================================================================
# FINE-TUNING DATASET CREATION v2 — FIXED
# Core logic: query → KG node search → variable resolution → decision
# ================================================================

!pip install networkx sentence-transformers tqdm ipywidgets -q

import json, os, pickle, random, re
from collections import defaultdict, Counter
import numpy as np
from tqdm import tqdm
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from sentence_transformers import SentenceTransformer
import networkx as nx

random.seed(42)
np.random.seed(42)

# ── Paths ─────────────────────────────────────────────────────
POPULATED_PATH = "/content/unified_dataset/unified_train_populated.json"
TRIMMED_PATH   = "/content/unified_dataset/unified_train_61k.json"
CKPT_PATH      = "/content/drive/MyDrive/kg_project/"
FT_OUT_DIR     = "/content/ft_dataset"
os.makedirs(FT_OUT_DIR, exist_ok=True)

# ── Load dataset ──────────────────────────────────────────────
DATASET_PATH = (POPULATED_PATH
                if os.path.exists(POPULATED_PATH)
                else TRIMMED_PATH)

print(f"Loading: {DATASET_PATH}")
with open(DATASET_PATH) as f:
    all_samples = json.load(f)

if DATASET_PATH == POPULATED_PATH:
    with open(TRIMMED_PATH) as f:
        trimmed_ids = set(s["id"] for s in json.load(f))
    all_samples = [s for s in all_samples if s["id"] in trimmed_ids]

print(f"Total: {len(all_samples)}")
print(f"Actions: {dict(Counter(s['action'] for s in all_samples))}")
print(f"Sources: {dict(Counter(s['metadata']['source'] for s in all_samples))}")


# ================================================================
# KG + SBERT SETUP
# ================================================================

class KnowledgeGraph:
    def __init__(self):
        self.G                = nx.DiGraph()
        self.kb_to_nodes      = defaultdict(list)
        self.kb_meta          = {}
        self.node_to_kbs      = defaultdict(set)
        self.variable_nodes   = set()
        self.reinforced_paths = {}

    def add_node(self, node_id, **attrs):
        if not self.G.has_node(node_id):
            self.G.add_node(node_id, **attrs)
        else:
            self.G.nodes[node_id].update(attrs)

    def add_edge(self, u, v, **attrs):
        if self.G.has_edge(u, v):
            ex = self.G[u][v]
            ex["weight"]    = max(ex.get("weight",0), attrs.get("weight",0))
            ex["frequency"] = ex.get("frequency",1) + 1
        else:
            attrs.setdefault("weight", 1.0)
            attrs.setdefault("frequency", 1)
            self.G.add_edge(u, v, **attrs)


def load_checkpoint(phase, path=CKPT_PATH):
    for fname in [f"kg_{phase}_clean.pkl", f"kg_{phase}.pkl"]:
        fpath = os.path.join(path, fname)
        if os.path.exists(fpath):
            with open(fpath, "rb") as f:
                data = pickle.load(f)
            kg = KnowledgeGraph()
            kg.G                = data["graph"]
            kg.kb_to_nodes      = defaultdict(list, data["kb_to_nodes"])
            kg.kb_meta          = data["kb_meta"]
            kg.node_to_kbs      = defaultdict(set, data["node_to_kbs"])
            kg.variable_nodes   = data["variable_nodes"]
            kg.reinforced_paths = data["reinforced_paths"]
            print(f"✅ Loaded {phase} ← {fpath}")
            print(f"   Nodes: {kg.G.number_of_nodes()}, "
                  f"Edges: {kg.G.number_of_edges()}")
            return kg
    print(f"❌ No checkpoint found for {phase} in {path}")
    print(f"   Available: {os.listdir(path)}")
    return None


print("\nLoading KG...")
kg = load_checkpoint("G2") or load_checkpoint("G1")

print("\nLoading SBERT...")
sbert = SentenceTransformer("all-MiniLM-L6-v2")

# ── Pre-encode all KG nodes ───────────────────────────────────
if kg is not None:
    all_nodes = list(kg.G.nodes())
    print(f"Encoding {len(all_nodes)} nodes...")
    node_matrix = sbert.encode(
        all_nodes,
        batch_size=512, show_progress_bar=True,
        convert_to_numpy=True, normalize_embeddings=True
    )
    G_undir = kg.G.to_undirected()
    print("✅ Node encoding done.")
else:
    all_nodes   = []
    node_matrix = None
    G_undir     = None


# ================================================================
# CACHE BUILDING — batch everything upfront
# ================================================================

MAX_TRIPLES        = 12
HOP_RADIUS         = 2
NODE_MATCH_THRESH  = 0.55
TRIPLE_REL_THRESH  = 0.35
MAX_VAR_SHOWN      = 3

print("\n⚡ Building caches...")

# 1. Collect all unique variable strings + queries
all_var_strings = set()
all_query_strings = set()
for s in all_samples:
    all_query_strings.add(s["query"])
    for v in s["state"].get("known_variables", []):
        if v: all_var_strings.add(v)
    for v in s["state"].get("missing_variables", []):
        if v: all_var_strings.add(v)

all_var_list   = list(all_var_strings)
all_query_list = list(all_query_strings)
print(f"   Unique variables : {len(all_var_list)}")
print(f"   Unique queries   : {len(all_query_list)}")

# 2. Batch encode variables
print("   Encoding variables...")
var_embs = sbert.encode(
    all_var_list, batch_size=512, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True
)

# ── Pre-score all variable strings for specificity ────────────
print("   Pre-scoring variable specificity …")
GENERIC_ANCHORS = [
    "clarification needed",
    "more information required",
    "unspecified context",
    "further details",
    "additional context needed",
    "insufficient information",
]
generic_anchor_embs = sbert.encode(
    GENERIC_ANCHORS, normalize_embeddings=True,
    convert_to_numpy=True, show_progress_bar=False
)  # (6, 384)

_specificity_cache = {}
SPECIFICITY_THRESHOLD = 0.45

if all_var_list:
    sims_to_generic = var_embs @ generic_anchor_embs.T  # (N_vars, 6)
    for var, sim_row in zip(all_var_list, sims_to_generic):
        _specificity_cache[var] = float(sim_row.max()) < SPECIFICITY_THRESHOLD

print(f"   Specific vars : "
      f"{sum(_specificity_cache.values())}/{len(_specificity_cache)}")

# 3. Batch encode queries
print("   Encoding queries...")
query_embs = sbert.encode(
    all_query_list, batch_size=512, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True
)

# 4. Build var→KG nodes map
print("   Mapping variables → KG nodes...")
_var_to_nodes = {}
if node_matrix is not None:
    CHUNK = 1000
    for i in tqdm(range(0, len(all_var_list), CHUNK), desc="   Var→node"):
        batch_v = all_var_list[i:i+CHUNK]
        batch_e = var_embs[i:i+CHUNK]
        sims    = batch_e @ node_matrix.T
        for j, var in enumerate(batch_v):
            row     = sims[j]
            top_idx = row.argsort()[::-1][:5]
            _var_to_nodes[var] = [
                (all_nodes[k], float(row[k]))
                for k in top_idx
                if float(row[k]) >= NODE_MATCH_THRESH
            ]
else:
    for v in all_var_list:
        _var_to_nodes[v] = []

# 5. Build query→KG nodes map
print("   Mapping queries → KG nodes...")
_query_to_nodes = {}
if node_matrix is not None:
    CHUNK = 1000
    for i in tqdm(range(0, len(all_query_list), CHUNK), desc="   Query→node"):
        batch_q = all_query_list[i:i+CHUNK]
        batch_e = query_embs[i:i+CHUNK]
        sims    = batch_e @ node_matrix.T
        for j, q in enumerate(batch_q):
            row     = sims[j]
            top_idx = row.argsort()[::-1][:5]
            _query_to_nodes[q] = [
                (all_nodes[k], float(row[k]))
                for k in top_idx
                if float(row[k]) >= NODE_MATCH_THRESH
            ]
else:
    for q in all_query_list:
        _query_to_nodes[q] = []

# 6. Query→embedding for triple filtering
_query_emb_map = {q: query_embs[i]
                  for i, q in enumerate(all_query_list)}

# 7. Pre-build ego caches for all seed nodes
all_seed_nodes = set()
for matches in list(_var_to_nodes.values()) + list(_query_to_nodes.values()):
    for node, _ in matches:
        all_seed_nodes.add(node)

print(f"   Building ego caches for {len(all_seed_nodes)} seed nodes...")
_ego_cache = {}
if G_undir is not None:
    for node in tqdm(all_seed_nodes, desc="   Ego graphs"):
        try:
            ego = nx.ego_graph(G_undir, node, radius=HOP_RADIUS)
            _ego_cache[node] = set(ego.nodes())
        except nx.NodeNotFound:
            _ego_cache[node] = {node}

# 8. Pre-compute triples per seed node
print("   Pre-computing triples per seed node...")
_node_triple_cache = {}
if kg is not None:
    for sn in tqdm(all_seed_nodes, desc="   Node triples"):
        ego_nodes = _ego_cache.get(sn, {sn})
        subG      = kg.G.subgraph(ego_nodes)
        edges     = sorted(
            subG.edges(data=True),
            key=lambda x: x[2].get("final_weight",
                                    x[2].get("weight", 0)),
            reverse=True
        )
        triples = []
        seen    = set()
        for u, v, d in edges[:MAX_TRIPLES * 3]:
            rel  = d.get("relation","related_to").replace("_"," ")
            line = f"{u} | {rel} | {v}"
            if line not in seen:
                seen.add(line)
                triples.append(line)
            if len(triples) >= MAX_TRIPLES * 2:
                break
        _node_triple_cache[sn] = triples

# 9. Variable-node adjacency for ?var lookup
_var_node_adj = {}
if kg is not None:
    for vn in kg.variable_nodes:
        _var_node_adj[vn] = (set(kg.G.predecessors(vn)) |
                             set(kg.G.successors(vn)))

print("✅ All caches built.\n")


# ================================================================
# CORE HELPERS
# ================================================================


def is_specific_missing(text: str) -> bool:
    """
    Returns True if the missing variable string has real semantic content.
    Uses precomputed cache — O(1) during main loop.
    Falls back to live sbert only for strings not seen during cache build.
    """
    if not text or len(text.strip()) < 4:
        return False
    if text in _specificity_cache:
        return _specificity_cache[text]
    # live fallback for unseen strings
    emb = sbert.encode(
        [text], normalize_embeddings=True,
        convert_to_numpy=True, show_progress_bar=False
    )[0]
    sim = float((generic_anchor_embs @ emb).max())
    result = sim < SPECIFICITY_THRESHOLD
    _specificity_cache[text] = result
    return result

def get_kg_nodes_for_var(var_str, top_k=3):
    return _var_to_nodes.get(var_str, [])[:top_k]


def get_kg_nodes_for_query(query_str, top_k=5):
    return _query_to_nodes.get(query_str, [])[:top_k]


def anonymize_var_nodes(triples):
    counter  = {}
    cleaned  = []
    for t in triples:
        parts = t.split(" | ")
        if len(parts) != 3:
            cleaned.append(t)
            continue
        new_parts = []
        for part in parts:
            if re.match(r"^\?var_", part):
                if part not in counter:
                    counter[part] = f"?unknown_{len(counter)+1}"
                new_parts.append(counter[part])
            else:
                new_parts.append(part)
        cleaned.append(" | ".join(new_parts))
    return cleaned


def filter_triples_by_relevance(query, triples, threshold=TRIPLE_REL_THRESH):
    if not triples:
        return []
    q_emb = _query_emb_map.get(query)
    if q_emb is None:
        return triples[:MAX_TRIPLES]

    triple_embs = sbert.encode(
        triples, normalize_embeddings=True, convert_to_numpy=True
    )
    sims   = q_emb @ triple_embs.T
    scored = sorted(zip(sims, triples), reverse=True)

    # only keep triples that actually score above threshold
    filtered = [t for s, t in scored if float(s) >= threshold]

    # if nothing passes — return empty, don't force top-3
    return filtered[:MAX_TRIPLES]


def get_subgraph_triples(query, known_variables, missing_variables,
                          action):
    if kg is None or not all_nodes:
        return [], [], []

    seed_nodes    = set()
    matched_nodes = []

    for node, score in get_kg_nodes_for_query(query, top_k=3):
        seed_nodes.add(node)
        matched_nodes.append(("query", node, score))

    for var in known_variables:
        for node, score in get_kg_nodes_for_var(var, top_k=2):
            seed_nodes.add(node)
            matched_nodes.append((var, node, score))

    if not seed_nodes:
        return [], [], []

    raw_triples = []
    seen        = set()
    for sn in seed_nodes:
        for line in _node_triple_cache.get(sn, []):
            if line not in seen:
                seen.add(line)
                raw_triples.append(line)

    var_triples = []
    if action == "ASK":
        var_count = 0
        for vn, adj_set in _var_node_adj.items():
            if var_count >= MAX_VAR_SHOWN:
                break
            hit = adj_set & seed_nodes
            if hit:
                line = f"{next(iter(hit))} | requires | {vn}"
                if line not in seen:
                    seen.add(line)
                    var_triples.append(line)
                    var_count += 1

    all_triples = raw_triples + var_triples
    filtered    = filter_triples_by_relevance(query, all_triples)
    anonymized  = anonymize_var_nodes(filtered)

    return anonymized, matched_nodes, list(seed_nodes)


def resolve_missing_from_graph(graph_triples, missing_variables):
    if missing_variables:
        return missing_variables

    inferred = []
    for t in graph_triples:
        parts = t.split(" | ")
        if len(parts) == 3 and "requires" in parts[1]:
            obj = parts[2].strip()
            if obj.startswith("?unknown"):
                inferred.append("clarification needed")
    return list(set(inferred)) if inferred else []


def summarize_graph_for_reasoning(graph_triples, matched_nodes, action):
    if not graph_triples:
        return ("The graph context contains no relevant connections "
                "for the entities in this query.")

    entities  = []
    relations = []
    for t in graph_triples[:6]:
        parts = t.split(" | ")
        if len(parts) == 3:
            subj, rel, obj = parts
            if not subj.startswith("?") and len(subj) > 2:
                entities.append(subj.strip())
            if not obj.startswith("?") and len(obj) > 2:
                entities.append(obj.strip())
            if (rel.strip() not in {"requires","related to"}
                    and len(rel) > 3):
                relations.append(rel.strip())

    entities  = list(dict.fromkeys(entities))[:4]
    relations = list(dict.fromkeys(relations))[:3]

    match_summary = ""
    if matched_nodes:
        top_matches = [(var, node) for var, node, sc in matched_nodes
                       if sc >= NODE_MATCH_THRESH][:3]
        if top_matches:
            parts = [f"'{node}' (matched from '{var}')"
                     for var, node in top_matches]
            match_summary = (f"Query terms matched KG nodes: "
                             f"{'; '.join(parts)}. ")

    entity_str   = (", ".join(entities)
                    if entities else "the query topic")
    relation_str = (", ".join(relations)
                    if relations else "various relations")

    if action == "ANSWER":
        return (
            f"{match_summary}"
            f"Graph traversal found connected nodes involving: {entity_str}. "
            f"Key relations present: {relation_str}. "
            f"The path from known entities to an answer is complete."
        )
    elif action == "ASK":
        requires_present = any("requires" in t for t in graph_triples)
        req_note = (" Variable placeholder nodes (requires edges) indicate "
                    "missing information in the graph."
                    if requires_present else "")
        return (
            f"{match_summary}"
            f"Graph traversal found partial connections involving: "
            f"{entity_str}. "
            f"Relations seen: {relation_str}.{req_note} "
            f"The path cannot be completed without additional information."
        )
    else:
        return (
            f"{match_summary}"
            f"Graph traversal found connections involving: {entity_str}, "
            f"but these do not connect to information that resolves the query. "
            f"The missing information is not obtainable through clarification."
        )


# ================================================================
# FIX 1 — Cumulative variable propagation across multi-turn
# ================================================================

def accumulate_variables_from_history(samples_in_dialogue,
                                       current_turn_id):
    cumulative_known    = set()
    cumulative_resolved = set()
    cumulative_missing  = set()

    prior_turns = [s for s in samples_in_dialogue
                   if (s["metadata"].get("turn_id") or 0)
                   < current_turn_id]

    for s in prior_turns:
        for v in s["state"].get("known_variables", []):
            if v:
                cumulative_known.add(v)

        s_missing = s["state"].get("missing_variables", [])
        s_action  = s["action"]

        if s_action == "ASK" and s_missing:
            cumulative_resolved.add(s_missing[0])
            cumulative_known.add(s_missing[0])

        for v in s_missing:
            if v:
                cumulative_missing.add(v)

    still_missing = cumulative_missing - cumulative_resolved

    return (sorted(cumulative_known),
            sorted(cumulative_resolved),
            sorted(still_missing))


# ================================================================
# FIX 2 — Better missing variable inference
# ================================================================

def infer_effective_missing(query, known_variables,
                             missing_variables, graph_triples,
                             action, cumulative_known=None,
                             still_missing_from_history=None):

    def _finalize(results):
        """Filter out generic placeholders before returning."""
        specific = [r for r in results if is_specific_missing(r)]
        return specific if specific else []

    # Step 1: use populated missing vars if available
    if missing_variables:
        resolved = set(cumulative_known or [])
        filtered = [m for m in missing_variables if m not in resolved]
        if filtered:
            return _finalize(filtered)

    # Step 2: use history-derived still-missing
    if still_missing_from_history:
        filtered = [m for m in still_missing_from_history
                    if m not in set(cumulative_known or [])]
        if filtered:
            return _finalize(filtered)

    # Step 3: infer from ?unknown requires edges in graph
    requires_subjects = []
    for t in graph_triples:
        parts = t.split(" | ")
        if (len(parts) == 3
                and "requires" in parts[1]
                and parts[2].startswith("?unknown")
                and not parts[0].startswith("?")):
            subj = parts[0].strip()
            if subj not in requires_subjects:
                requires_subjects.append(subj)

    if requires_subjects and action in ("ASK", "ABSTAIN"):
        return _finalize([f"details about '{requires_subjects[0]}'"])

    # Step 4: infer from query semantics
    q_lower = query.lower()
    if action == "ASK":
        if q_lower.startswith("what"):
            words = query.split()
            if len(words) > 1:
                return _finalize([f"'{words[1]}' information"])
        if any(w in q_lower for w in
               ["who","which","where","when","how"]):
            word = next(w for w in
                        ["who","which","where","when","how"]
                        if w in q_lower)
            return _finalize([f"'{word}' referent for this query"])
        short_q = query[:40].rstrip("?").strip()
        return _finalize([f"context for: '{short_q}'"])

    if action == "ABSTAIN":
        specific = [k for k in (known_variables or [])
                    if is_specific_missing(k)]
        topic = specific[0] if specific else (
            known_variables[0] if known_variables
            else query.split()[0]
        )
        return _finalize(
            [f"information about '{topic}' absent from knowledge base"]
        )

    return []


# ================================================================
# FIX 3 — Cap ?unknown requires edges in graph context
# ================================================================

def cap_requires_edges(triples, max_requires=3, action="ANSWER"):
    if action == "ANSWER":
        # ANSWER samples must never show requires/?unknown edges
        return [t for t in triples
                if not ("requires" in t and "?unknown" in t)]

    # ASK / ABSTAIN — keep but cap
    requires, others = [], []
    for t in triples:
        parts = t.split(" | ")
        if (len(parts) == 3
                and "requires" in parts[1]
                and "?unknown" in parts[2]):
            requires.append(t)
        else:
            others.append(t)
    return others + requires[:max_requires]


# ================================================================
# FIX 4 — ASK response should be a real question
# ================================================================

def build_ask_question(query, effective_missing,
                        known_variables, graph_triples,
                        cumulative_known=None):
    if not effective_missing:
        subject = (known_variables[0] if known_variables
                   else query.split()[0] if query else "this topic")
        return f"Could you provide more details about {subject}?"

    top_missing   = effective_missing[0]
    clean_missing = re.sub(
        r"^(details about|context for:|information about)\s*",
        "", top_missing, flags=re.IGNORECASE
    ).strip("'\" ")

    # ── Step 1: try known_variables first ─────────────────────────
    # known variables are ground truth — always prefer them
    anchor = None
    if known_variables and clean_missing:
        kv_embs = sbert.encode(
            [clean_missing] + known_variables,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False
        )
        miss_emb = kv_embs[0]
        kv_only  = kv_embs[1:]
        sims     = kv_only @ miss_emb
        best_idx = int(sims.argmax())
        best_sim = float(sims[best_idx])
        if best_sim >= 0.20:   # low bar — known vars are trusted
            anchor = known_variables[best_idx]

    # ── Step 2: only if no known var matched, try graph subjects ──
    if not anchor:
        graph_subjects = []
        for t in graph_triples:
            parts = t.split(" | ")
            if (len(parts) == 3
                    and not parts[0].startswith("?")
                    and "requires" not in parts[1]
                    and len(parts[0].strip()) > 2):
                graph_subjects.append(parts[0].strip())

        graph_subjects = list(dict.fromkeys(graph_subjects))

        if graph_subjects and clean_missing:
            gs_embs  = sbert.encode(
                [clean_missing] + graph_subjects,
                normalize_embeddings=True,
                convert_to_numpy=True,
                show_progress_bar=False
            )
            miss_emb = gs_embs[0]
            gs_only  = gs_embs[1:]
            sims     = gs_only @ miss_emb
            best_idx = int(sims.argmax())
            best_sim = float(sims[best_idx])
            # higher bar for graph nodes — less trusted
            if best_sim >= 0.45:
                anchor = graph_subjects[best_idx]

    # ── Step 3: hard fallback — first known variable ───────────────
    if not anchor:
        anchor = known_variables[0] if known_variables else None

    if not anchor:
        return f"Could you specify {clean_missing}?"

    if anchor.lower() == clean_missing.lower():
        return f"Could you provide more details about {anchor}?"

    return f"Regarding {anchor}: could you specify {clean_missing}?"

# ================================================================
# CONVERSATION HISTORY BUILDER
# ================================================================

DUMMY_RESPONSES = {
    "Could you provide more details so I can give a more precise answer?",
    "I do not have enough information to answer this question.",
    "I do not have enough information to determine this.",
}


def clean_response_for_history(response, action, missing_variables):
    is_dummy = response.strip() in DUMMY_RESPONSES

    if not is_dummy:
        return response[:120]

    if action == "ASK":
        if missing_variables:
            top_missing = missing_variables[0]
            return (f"[Clarification requested: "
                    f"'{top_missing}' is needed to proceed]")
        return "[Clarification requested to resolve the query]"

    if action == "ABSTAIN":
        if missing_variables:
            top_missing = missing_variables[0]
            return (f"[Cannot answer: '{top_missing}' "
                    f"is absent from available context]")
        return "[Cannot answer: insufficient context]"

    return response[:120]


def build_conversation_history(samples_in_dialogue, current_turn_id):
    history = []
    cumulative_resolved = set()

    for s in samples_in_dialogue:
        t_id = s["metadata"].get("turn_id") or 0
        if t_id >= current_turn_id:
            break

        s_known   = s["state"].get("known_variables",   [])
        s_missing = s["state"].get("missing_variables", [])
        s_action  = s["action"]
        s_response = s["response"]

        clean_resp = clean_response_for_history(
            s_response, s_action, s_missing
        )

        resolved_var = None
        if s_action == "ASK" and s_missing:
            resolved_var = s_missing[0]
            cumulative_resolved.add(resolved_var)

        history.append({
            "turn_id"          : t_id,
            "query"            : s["query"],
            "action"           : s_action,
            "response"         : clean_resp,
            "missing_variables": s_missing,
            "resolved_variable": resolved_var,
        })

    return history if history else None, cumulative_resolved


# ================================================================
# PROMPT BUILDERS
# ================================================================

SYSTEM_PROMPT = """You are a decision planner for a question-answering system.

Your task: given a user query, search the knowledge graph for relevant nodes, evaluate what information is present and what is missing, then decide the correct action.

Decision logic:
- Search the graph for nodes matching the query subject and known variables
- If the graph contains a complete path connecting known entities to an answer → ANSWER
- If the graph contains the topic but key linking variables are missing → ASK (specify what is missing)
- If the graph has no relevant nodes or the topic is entirely absent → ABSTAIN

You will receive:
<query> — the user's question
<known_variables> — entities explicitly present in the query
<graph_context> — KG triples from nodes matching the query (subject | relation | object)
<missing_variables> — variables required but not present
<conversation_history> — prior turns (for multi-turn queries only)

Output format (strictly follow this):
<reasoning>
Step 1 — Query subject: identify what the query is asking about
Step 2 — Graph search: what nodes were found, what connections exist
Step 3 — Variable check: what is known, what is missing
Step 4 — Decision rationale: why this action is correct
</reasoning>

<decision>
ANSWER | ASK | ABSTAIN
</decision>

<justification>
One sentence grounded in the graph evidence.
</justification>

Rules:
- Reasoning must reference actual graph content, not generic statements
- Never say "unspecified variables" — name the specific missing variable
- If graph_context is empty, default to ABSTAIN unless context is clearly partial (then ASK)
- Do not use prior world knowledge — only the graph context provided"""


# ================================================================
# FIX 7 — Updated build_user_turn signature
# ================================================================

def build_user_turn(sample, graph_triples, history=None,
                    cumulative_resolved=None,
                    cumulative_known=None, all_known=None):

    known   = all_known or sample["state"].get("known_variables", [])
    missing = sample["state"].get("missing_variables", [])

    effective_missing = resolve_missing_from_graph(
        graph_triples, missing
    )
    if cumulative_resolved:
        effective_missing = [m for m in effective_missing
                             if m not in cumulative_resolved]

    # History block
    history_block = ""
    if history:
        history_block = "<conversation_history>\n"
        for h in history:
            history_block += (
                f"Turn {h['turn_id']} | {h['action']} | "
                f"Q: \"{h['query'][:80]}\" | "
                f"A: {h['response']}\n"
            )
            if h.get("resolved_variable"):
                history_block += (
                    f"  → resolved: '{h['resolved_variable']}'\n"
                )
        history_block += "</conversation_history>\n\n"

    # State block for multi-turn
    state_block = ""
    if history:
        resolved      = cumulative_resolved or set()
        still_missing = [
            m for m in effective_missing
            if m not in resolved
            and is_specific_missing(m)   # ← semantic filter, no word list
        ]
        if still_missing:
            state_block += (
                "<remaining_unknowns>\n"
                + "\n".join(f"- {m}" for m in still_missing)
                + "\n</remaining_unknowns>\n"
            )
        if resolved:
            state_block += (
                "<resolved_variables>\n"
                + "\n".join(f"- {r}" for r in sorted(resolved))
                + "\n</resolved_variables>\n"
            )
        if state_block:
            state_block += "\n"

    # Graph context
    graph_block = (
        "<graph_context>\n"
        + "\n".join(graph_triples)
        + "\n</graph_context>"
    ) if graph_triples else (
        "<graph_context>\n"
        "No relevant nodes found in knowledge graph.\n"
        "</graph_context>"
    )

    # Missing block
    missing_block = (
        "<missing_variables>\n"
        + "\n".join(f"- {m}" for m in effective_missing)
        + "\n</missing_variables>"
    ) if effective_missing else (
        "<missing_variables>\nnone\n</missing_variables>"
    )

    user_turn = (
        f"{history_block}"
        f"{state_block}"
        f"<query>\n{sample['query']}\n</query>\n\n"
        f"<known_variables>\n"
        f"{', '.join(known) if known else 'none identified'}\n"
        f"</known_variables>\n\n"
        f"{graph_block}\n\n"
        f"{missing_block}\n\n"
        f"Search the graph context for relevant nodes and decide "
        f"the correct action."
    )

    return user_turn, effective_missing


# ================================================================
# FIX 5 — Updated build_assistant_turn using all fixes
# ================================================================

def build_assistant_turn(sample, graph_triples, matched_nodes,
                          effective_missing, cumulative_known=None):
    action = sample["action"]
    known  = sample["state"].get("known_variables", [])
    fm     = sample["state"].get("failure_mode", "COMPLETE")

    all_known = list(set(known + (cumulative_known or [])))

    query_subject = (", ".join(all_known[:3])
                     if all_known else f"'{sample['query'][:50]}'")

    graph_summary = summarize_graph_for_reasoning(
        graph_triples, matched_nodes, action
    )

    if action == "ANSWER":
        var_check = (
            f"Known variables ({', '.join(all_known[:3]) if all_known else 'query entities'}) "
            f"are present in the graph. "
            f"No critical variables are missing. "
            f"Failure mode: {fm} (complete information state)."
        )
        decision_rationale = (
            "The graph provides a complete reasoning path "
            "from the query entities to a resolvable answer."
        )
        justification = (
            "Graph traversal is complete — all required nodes "
            "are connected and no missing variables block the answer."
        )
        response_line = ""

    elif action == "ASK":
        missing_str = (
            ", ".join(f"'{m}'" for m in effective_missing[:2])
            if effective_missing else "'further context'"
        )
        var_check = (
            f"Known: {', '.join(all_known[:3]) if all_known else 'query entities only'}. "
            f"Required but absent from graph: {missing_str}. "
            f"Failure mode: {fm}."
        )
        decision_rationale = (
            f"The graph has partial connections for this topic but "
            f"cannot complete the reasoning path without: {missing_str}."
        )
        ask_q = build_ask_question(
            sample["query"], effective_missing,
            all_known, graph_triples, cumulative_known
        )
        justification = ask_q
        response_line = f"\n\n<clarification_question>\n{ask_q}\n</clarification_question>"

    else:  # ABSTAIN
        missing_str = (
            ", ".join(f"'{m}'" for m in effective_missing[:2])
            if effective_missing else "'required context'"
        )
        var_check = (
            f"Known: {', '.join(all_known[:3]) if all_known else 'none'}. "
            f"Missing: {missing_str}. "
            f"Failure mode: {fm} — "
            f"information is absent from the graph entirely."
        )
        decision_rationale = (
            f"The graph lacks nodes to resolve this query. "
            f"Missing: {missing_str}. "
            f"No clarification from the user can fill this gap."
        )
        justification = (
            f"Graph has no resolvable path — "
            f"{missing_str} is entirely absent from the knowledge base."
        )
        response_line = ""

    assistant_turn = (
        f"<reasoning>\n"
        f"Step 1 — Query subject: {query_subject}. "
        f"Query asks: '{sample['query'][:80]}'\n"
        f"Step 2 — Graph search: {graph_summary}\n"
        f"Step 3 — Variable check: {var_check}\n"
        f"Step 4 — Decision rationale: {decision_rationale}\n"
        f"</reasoning>\n\n"
        f"<decision>\n{action}\n</decision>\n\n"
        f"<justification>\n{justification}\n</justification>"
        f"{response_line}"
    )

    return assistant_turn


# ================================================================
# SAMPLE QUALITY FILTER
# ================================================================

def is_valid_training_sample(known, graph_triples, action, effective_missing):
    if not known and not graph_triples and action == "ASK":
        return False, "zero_signal_ask"

    if not graph_triples and action == "ANSWER" and not known:
        return False, "no_graph_answer"

    if (action == "ASK"
            and not effective_missing
            and not any("requires" in t for t in graph_triples)):
        return False, "ask_no_missing_no_var_nodes"

    # NEW — ANSWER: at least one triple must contain a known entity
    if action == "ANSWER" and graph_triples and known:
        known_lower = {k.lower().strip() for k in known if k}
        has_match   = any(
            any(k in t.lower() for k in known_lower)
            for t in graph_triples
        )
        if not has_match:
            return False, "answer_graph_irrelevant"

    # NEW — ANSWER with empty graph after filtering → ABSTAIN instead
    if action == "ANSWER" and not graph_triples:
        return False, "answer_empty_graph"

    return True, "ok"


# ================================================================
# FIX 6 — Updated make_ft_sample with full variable propagation
# ================================================================

def make_ft_sample(s, graph_triples, matched_nodes, seed_nodes,
                   effective_missing, history, cum_resolved,
                   ft_id, cumulative_known=None,
                   still_missing_from_history=None):

    all_known = list(set(
        s["state"].get("known_variables", [])
        + (cumulative_known or [])
    ))

    user_turn, _ = build_user_turn(
        s, graph_triples, history, cum_resolved,
        cumulative_known=cumulative_known,
        all_known=all_known
    )
    asst_turn = build_assistant_turn(
        s, graph_triples, matched_nodes,
        effective_missing,
        cumulative_known=all_known
    )

    return {
        "ft_id"              : ft_id,
        "source_id"          : s["id"],
        "source"             : s["metadata"]["source"],
        "action"             : s["action"],
        "difficulty"         : s["state"].get("difficulty","medium"),
        "failure_mode"       : s["state"].get("failure_mode",""),
        "multi_turn"         : s["metadata"].get("multi_turn", False),
        "turn_id"            : s["metadata"].get("turn_id"),
        "dialogue_id"        : s["metadata"].get("dialogue_id"),
        "num_known"          : len(all_known),
        "num_missing"        : len(effective_missing),
        "num_triples"        : len(graph_triples),
        "num_matched_nodes"  : len(seed_nodes),
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": user_turn},
            {"role": "assistant", "content": asst_turn},
        ],
        "_debug": {
            "query"                     : s["query"],
            "known_variables"           : s["state"].get("known_variables",[]),
            "cumulative_known"          : cumulative_known or [],
            "all_known_this_turn"       : all_known,
            "missing_variables"         : s["state"].get("missing_variables",[]),
            "still_missing_from_history": still_missing_from_history or [],
            "effective_missing"         : effective_missing,
            "graph_triples"             : graph_triples,
            "matched_nodes"             : [(v,n,round(sc,3))
                                           for v,n,sc in matched_nodes[:5]],
            "seed_nodes"                : list(seed_nodes)[:10],
            "original_response"         : s["response"],
            "history_length"            : len(history) if history else 0,
            "resolved_vars"             : list(cum_resolved),
            "context_preview"           : (
                s["context"]["documents"][0]["text"][:200]
                if s["context"]["documents"] else ""
            ),
        }
    }


# ================================================================
# MAIN CONSTRUCTION LOOP
# ================================================================

print("\n" + "="*60)
print("  BUILDING FINE-TUNING DATASET v2")
print("="*60)

dialogues   = defaultdict(list)
single_turn = []

for s in all_samples:
    if (s["metadata"].get("multi_turn")
            and s["metadata"].get("dialogue_id")):
        dialogues[s["metadata"]["dialogue_id"]].append(s)
    else:
        single_turn.append(s)

for dlg_id in dialogues:
    dialogues[dlg_id].sort(
        key=lambda x: x["metadata"].get("turn_id") or 0
    )

print(f"Multi-turn dialogues : {len(dialogues)}")
print(f"Single-turn samples  : {len(single_turn)}")

ft_samples    = []
skipped       = Counter()
graph_miss    = 0
ft_id_counter = 0


# ── Single-turn ────────────────────────────────────────────────
print("\nProcessing single-turn samples...")
for s in tqdm(single_turn, desc="Single-turn"):
    known   = s["state"].get("known_variables",   [])
    missing = s["state"].get("missing_variables", [])
    action  = s["action"]

    graph_triples, matched_nodes, seed_nodes = get_subgraph_triples(
        s["query"], known, missing, action
    )

    # FIX 3: cap requires edges
    graph_triples = cap_requires_edges(graph_triples, max_requires=3, action=action)

    # FIX 2: use improved missing inference
    effective_missing = infer_effective_missing(
        s["query"], known, missing, graph_triples, action
    )

    valid, reason = is_valid_training_sample(
        known, graph_triples, action, effective_missing
    )
    if not valid:
        skipped[reason] += 1
        continue

    if not graph_triples:
        graph_miss += 1

    sample = make_ft_sample(
        s, graph_triples, matched_nodes, seed_nodes,
        effective_missing, None, set(),
        f"ft_{ft_id_counter:07d}"
    )
    ft_samples.append(sample)
    ft_id_counter += 1


# ── FIX 8 — Updated Multi-turn loop ───────────────────────────
print("\nProcessing multi-turn dialogues...")
for dlg_id, turns in tqdm(dialogues.items(), desc="Dialogues"):
    for turn in turns:
        t_id    = turn["metadata"].get("turn_id") or 0
        known   = turn["state"].get("known_variables",   [])
        missing = turn["state"].get("missing_variables", [])
        action  = turn["action"]

        # FIX 1: accumulate variables from ALL prior turns
        (cum_known,
         cum_resolved_list,
         still_missing_hist) = accumulate_variables_from_history(
            turns, t_id
        )
        cum_resolved = set(cum_resolved_list)

        all_known_this_turn = list(set(known + cum_known))

        history, _ = build_conversation_history(turns, t_id)

        graph_triples, matched_nodes, seed_nodes = get_subgraph_triples(
            turn["query"], all_known_this_turn, missing, action
        )

        # FIX 3: cap requires edges
        graph_triples = cap_requires_edges(graph_triples, max_requires=3, action=action)

        # FIX 2: improved missing inference with full context
        effective_missing = infer_effective_missing(
            turn["query"], all_known_this_turn, missing,
            graph_triples, action,
            cumulative_known=all_known_this_turn,
            still_missing_from_history=still_missing_hist
        )
        effective_missing = [m for m in effective_missing
                             if m not in cum_resolved]

        valid, reason = is_valid_training_sample(
            all_known_this_turn, graph_triples,
            action, effective_missing
        )
        if not valid:
            skipped[reason] += 1
            continue

        if not graph_triples:
            graph_miss += 1

        sample_obj = make_ft_sample(
            turn, graph_triples, matched_nodes, seed_nodes,
            effective_missing, history, cum_resolved,
            f"ft_{ft_id_counter:07d}",
            cumulative_known=cum_known,
            still_missing_from_history=still_missing_hist
        )
        ft_samples.append(sample_obj)
        ft_id_counter += 1


# ── Stats ──────────────────────────────────────────────────────
print(f"\n✅ Built {len(ft_samples)} fine-tuning samples")
print(f"   Skipped: {dict(skipped)}")
print(f"   No graph triples: {graph_miss} "
      f"({100*graph_miss/max(len(ft_samples),1):.1f}%)")

act_dist = Counter(s["action"] for s in ft_samples)
print(f"\n  Action distribution:")
for act in ["ANSWER","ASK","ABSTAIN"]:
    n = act_dist[act]
    print(f"  {act:10}: {n:6}  ({100*n/len(ft_samples):.1f}%)")

src_dist = Counter(s["source"] for s in ft_samples)
print(f"\n  Source distribution:")
for src, cnt in sorted(src_dist.items()):
    print(f"  {src:15}: {cnt:6}  ({100*cnt/len(ft_samples):.1f}%)")

matched_counts = [s["num_matched_nodes"] for s in ft_samples]
triple_counts  = [s["num_triples"] for s in ft_samples]
print(f"\n  Avg matched KG nodes : {np.mean(matched_counts):.2f}")
print(f"  Avg graph triples    : {np.mean(triple_counts):.2f}")
print(f"  Samples with 0 nodes : "
      f"{sum(1 for c in matched_counts if c==0)}")


# ================================================================
# TRAIN / VAL / TEST SPLIT — dialogue-level
# ================================================================

print("\n" + "="*60)
print("  TRAIN / VAL / TEST SPLIT")
print("="*60)

all_dlg_ids    = list(set(s["dialogue_id"] for s in ft_samples
                          if s["dialogue_id"]))
all_single_ids = [s["ft_id"] for s in ft_samples
                  if not s["dialogue_id"]]

random.shuffle(all_dlg_ids)
random.shuffle(all_single_ids)

n_dlg      = len(all_dlg_ids)
n_train_d  = int(0.70 * n_dlg)
n_val_d    = int(0.15 * n_dlg)

train_dlgs = set(all_dlg_ids[:n_train_d])
val_dlgs   = set(all_dlg_ids[n_train_d:n_train_d+n_val_d])
test_dlgs  = set(all_dlg_ids[n_train_d+n_val_d:])

n_single   = len(all_single_ids)
n_train_s  = int(0.70 * n_single)
n_val_s    = int(0.15 * n_single)

train_singles = set(all_single_ids[:n_train_s])
val_singles   = set(all_single_ids[n_train_s:n_train_s+n_val_s])
test_singles  = set(all_single_ids[n_train_s+n_val_s:])

train_samples, val_samples, test_samples = [], [], []

for s in ft_samples:
    if s["dialogue_id"]:
        if s["dialogue_id"] in train_dlgs:   train_samples.append(s)
        elif s["dialogue_id"] in val_dlgs:   val_samples.append(s)
        else:                                 test_samples.append(s)
    else:
        if s["ft_id"] in train_singles:       train_samples.append(s)
        elif s["ft_id"] in val_singles:       val_samples.append(s)
        else:                                 test_samples.append(s)

train_samples.sort(key=lambda x: (
    x["dialogue_id"] or x["ft_id"],
    x["turn_id"] or 0
))

print(f"\n  Train : {len(train_samples):6}")
print(f"  Val   : {len(val_samples):6}")
print(f"  Test  : {len(test_samples):6}")

for split_name, split in [("val",val_samples),("test",test_samples)]:
    leak = sum(1 for s in split
               if s["dialogue_id"] and s["dialogue_id"] in train_dlgs)
    print(f"  Leak check {split_name:5}: {leak} "
          f"({'✅' if leak==0 else '⚠️ LEAK'})")


# ================================================================
# SAVE
# ================================================================

def strip_debug(samples):
    return [{k: v for k, v in s.items() if k != "_debug"}
            for s in samples]

for name, split in [("train",train_samples),
                    ("val",  val_samples),
                    ("test", test_samples)]:
    p = f"{FT_OUT_DIR}/ft_{name}_debug.json"
    with open(p,"w") as f: json.dump(split, f, indent=2)
    print(f"Saved {name} debug   → {p} ({os.path.getsize(p)/1e6:.1f}MB)")

    p = f"{FT_OUT_DIR}/ft_{name}.jsonl"
    with open(p,"w") as f:
        for s in strip_debug(split):
            f.write(json.dumps(s)+"\n")
    print(f"Saved {name} JSONL   → {p} ({os.path.getsize(p)/1e6:.1f}MB)")

    p = f"{FT_OUT_DIR}/ft_{name}_mistral.jsonl"
    with open(p,"w") as f:
        for s in split:
            f.write(json.dumps({"messages": s["messages"]})+"\n")
    print(f"Saved {name} Mistral → {p} ({os.path.getsize(p)/1e6:.1f}MB)")


# ================================================================
# QUALITY STATS PER SPLIT
# ================================================================

def split_stats(samples, name):
    print(f"\n  ── {name} ({len(samples)}) ──")
    acts  = Counter(s["action"] for s in samples)
    srcs  = Counter(s["source"] for s in samples)
    mt    = sum(1 for s in samples if s["multi_turn"])
    trip  = [s["num_triples"]       for s in samples]
    kn    = [s["num_known"]          for s in samples]
    ms    = [s["num_missing"]        for s in samples]
    nd    = [s["num_matched_nodes"]  for s in samples]
    zero_t= sum(1 for t in trip if t==0)
    zero_n= sum(1 for n in nd   if n==0)

    for act in ["ANSWER","ASK","ABSTAIN"]:
        n = acts[act]
        print(f"    {act:10}: {n:5} ({100*n/max(len(samples),1):.1f}%)")

    print(f"    Multi-turn       : {mt} ({100*mt/max(len(samples),1):.1f}%)")
    print(f"    Avg triples      : {np.mean(trip):.2f} "
          f"[0-triple: {zero_t} ({100*zero_t/max(len(samples),1):.1f}%)]")
    print(f"    Avg matched nodes: {np.mean(nd):.2f} "
          f"[0-node: {zero_n} ({100*zero_n/max(len(samples),1):.1f}%)]")
    print(f"    Avg known vars   : {np.mean(kn):.2f}")
    print(f"    Avg missing vars : {np.mean(ms):.2f}")

    avg_u = np.mean([len(s["messages"][1]["content"].split())
                     for s in samples])
    avg_a = np.mean([len(s["messages"][2]["content"].split())
                     for s in samples])
    print(f"    Avg user words   : {avg_u:.0f}")
    print(f"    Avg asst words   : {avg_a:.0f}")

    print(f"    Sources: "
          + " | ".join(f"{k}:{v}" for k,v in sorted(srcs.items())))

print("\n" + "="*60)
print("  DATASET QUALITY STATS")
print("="*60)
split_stats(train_samples, "TRAIN")
split_stats(val_samples,   "VAL")
split_stats(test_samples,  "TEST")


# ================================================================
# INTERACTIVE DEBUG VIEWER
# ================================================================

ACT_COLOR = {"ANSWER":"#00e676","ASK":"#40c4ff","ABSTAIN":"#ff5252"}
SRC_COLOR = {"quac":"#80cbc4","sharc":"#ce93d8",
             "hotpotqa":"#ffcc80","contract_nli":"#f48fb1"}
SPLIT_MAP = {"train":train_samples,"val":val_samples,"test":test_samples}

split_dd  = widgets.Dropdown(
    options=["train","val","test"], value="train",
    description="Split:", style={"description_width":"60px"},
    layout=widgets.Layout(width="180px"))
action_dd = widgets.Dropdown(
    options=["all","ANSWER","ASK","ABSTAIN"], value="ASK",
    description="Action:", style={"description_width":"60px"},
    layout=widgets.Layout(width="180px"))
source_dd = widgets.Dropdown(
    options=["all","quac","sharc","hotpotqa","contract_nli"], value="all",
    description="Source:", style={"description_width":"60px"},
    layout=widgets.Layout(width="200px"))
mt_dd     = widgets.Dropdown(
    options=["all","multi-turn","single-turn"], value="all",
    description="Turn:", style={"description_width":"60px"},
    layout=widgets.Layout(width="180px"))
kg_dd     = widgets.Dropdown(
    options=["all","has triples","no triples","has matched nodes",
             "no matched nodes"],
    value="all", description="Graph:", style={"description_width":"60px"},
    layout=widgets.Layout(width="200px"))
view_dd   = widgets.ToggleButtons(
    options=["Formatted","Raw JSON","Mistral Prompt","History Detail"],
    description="View:", style={"description_width":"50px",
                                "button_width":"140px"})
idx_slider= widgets.IntSlider(
    value=0, min=0, max=100, step=1,
    description="Sample:", style={"description_width":"70px"},
    layout=widgets.Layout(width="550px"))
out = widgets.Output()


def get_pool(split, action, source, mt, kg_filter):
    pool = SPLIT_MAP[split]
    if action != "all":
        pool = [s for s in pool if s["action"] == action]
    if source != "all":
        pool = [s for s in pool if s["source"] == source]
    if mt == "multi-turn":
        pool = [s for s in pool if s["multi_turn"]]
    elif mt == "single-turn":
        pool = [s for s in pool if not s["multi_turn"]]
    if kg_filter == "has triples":
        pool = [s for s in pool if s["num_triples"] > 0]
    elif kg_filter == "no triples":
        pool = [s for s in pool if s["num_triples"] == 0]
    elif kg_filter == "has matched nodes":
        pool = [s for s in pool if s["num_matched_nodes"] > 0]
    elif kg_filter == "no matched nodes":
        pool = [s for s in pool if s["num_matched_nodes"] == 0]
    return pool


def render_formatted(s):
    d      = s["_debug"]
    action = s["action"]
    color  = ACT_COLOR.get(action,"#fff")
    src_c  = SRC_COLOR.get(s["source"],"#aaa")

    triple_rows = ""
    for t in d["graph_triples"]:
        parts = t.split(" | ")
        if len(parts) == 3:
            subj, rel, obj = parts
            subj_col = "#80cbc4"
            obj_col  = "#ce93d8"
            triple_rows += (
                f'<div style="padding:2px 0; font-size:11px;">'
                f'<span style="color:{subj_col};">{subj}</span>'
                f' <span style="color:#ffcc80;">│ {rel} │</span>'
                f' <span style="color:{obj_col};">{obj}</span>'
                f'</div>'
            )
        else:
            triple_rows += (
                f'<div style="color:#888; font-size:11px;">{t}</div>'
            )

    node_rows = ""
    for var, node, sc in d["matched_nodes"]:
        node_rows += (
            f'<div style="font-size:10px; color:#aaa; padding:1px 0;">'
            f'"{var}" → <span style="color:#80cbc4;">{node}</span> '
            f'<span style="color:#555;">(score={sc})</span></div>'
        )

    def tag_list(items, bg, fg):
        if not items:
            return '<span style="color:#555;">— none —</span>'
        return "".join(
            f'<span style="background:{bg}; color:{fg}; '
            f'padding:2px 8px; border-radius:4px; margin:2px; '
            f'display:inline-block; font-size:11px;">{v}</span>'
            for v in items
        )

    known_html   = tag_list(d["all_known_this_turn"],    "#1b3a2b","#00e676")
    missing_html = tag_list(d["missing_variables"],       "#3a1b1b","#ff5252")
    eff_html     = tag_list(d["effective_missing"],       "#2a1a3a","#ce93d8")

    asst = s["messages"][2]["content"]
    reasoning, decision_txt, justification = "", action, ""
    for tag in ["reasoning","decision","justification"]:
        s_tag = f"<{tag}>"; e_tag = f"</{tag}>"
        si = asst.find(s_tag); ei = asst.find(e_tag)
        if si != -1 and ei != -1:
            val = asst[si+len(s_tag):ei].strip()
            if tag == "reasoning":       reasoning     = val
            elif tag == "decision":      decision_txt  = val
            elif tag == "justification": justification = val

    reas_html = reasoning.replace(
        "Step 1","<b style='color:#90caf9;'>Step 1</b>"
    ).replace(
        "Step 2","<b style='color:#90caf9;'>Step 2</b>"
    ).replace(
        "Step 3","<b style='color:#90caf9;'>Step 3</b>"
    ).replace(
        "Step 4","<b style='color:#90caf9;'>Step 4</b>"
    ).replace("\n","<br>")

    history_note = (f"History: {d['history_length']} prior turns | "
                    f"Resolved: {d['resolved_vars']}"
                    if s["multi_turn"] else "Single-turn")

    return f"""
    <div style="font-family:monospace; font-size:12px; background:#1e1e1e;
                color:#e0e0e0; padding:16px; border-radius:10px;
                border:1px solid #444; line-height:1.5;">

      <div style="display:flex; gap:14px; flex-wrap:wrap; margin-bottom:8px;">
        <span><b style="color:#90caf9;">ID:</b> {s['ft_id']}</span>
        <span><b style="color:#90caf9;">Src:</b>
          <span style="color:{src_c};">{s['source']}</span></span>
        <span><b style="color:#90caf9;">Diff:</b>
          <span style="color:#ffcc80;">{s['difficulty']}</span></span>
        <span><b style="color:#90caf9;">FM:</b>
          <span style="color:#ffcc80;">{s['failure_mode']}</span></span>
        <span><b style="color:#90caf9;">Turn:</b>
          {"Turn "+str(s['turn_id']) if s['multi_turn'] else "Single"}</span>
        <span style="color:#aaa; font-size:10px;">{history_note}</span>
      </div>
      <hr style="border:0.5px solid #333;">

      <div style="background:#1a2632; padding:5px 8px; border-radius:4px;
                  font-size:10px; color:#78909c; margin-bottom:8px;">
        {d['context_preview']}...
      </div>

      <div style="margin-bottom:8px; font-size:13px;">
        <b style="color:#fff176;">Q:</b> {d['query']}
      </div>

      <div style="display:grid; grid-template-columns:1fr 1fr 1fr;
                  gap:10px; margin-bottom:10px;">
        <div>
          <b style="color:#90caf9; font-size:11px;">All Known
            ({len(d['all_known_this_turn'])}):</b><br>
          {known_html}
        </div>
        <div>
          <b style="color:#90caf9; font-size:11px;">Missing
            ({len(d['missing_variables'])}):</b><br>
          {missing_html}
        </div>
        <div>
          <b style="color:#90caf9; font-size:11px;">Effective missing
            ({len(d['effective_missing'])}):</b><br>
          {eff_html}
        </div>
      </div>

      <div style="margin-bottom:8px;">
        <b style="color:#90caf9; font-size:11px;">
          🔍 KG Node Matches ({s['num_matched_nodes']}):</b>
        <div style="background:#151f2e; padding:5px 8px; border-radius:4px;
                    margin-top:3px;">
          {node_rows if node_rows
           else '<span style="color:#555; font-size:10px;">no nodes matched</span>'}
        </div>
      </div>

      <div style="margin-bottom:10px;">
        <b style="color:#90caf9; font-size:11px;">
          📊 Graph Context ({s['num_triples']} triples):</b>
        <div style="background:#151f2e; padding:8px; border-radius:4px;
                    margin-top:3px; max-height:140px; overflow-y:auto;">
          {triple_rows if triple_rows
           else '<span style="color:#555; font-size:10px;">no triples found</span>'}
        </div>
      </div>
      <hr style="border:0.5px solid #333;">

      <div style="margin-bottom:6px;">
        <b style="color:#90caf9;">Decision:</b>
        <span style="color:{color}; font-weight:bold;
                     font-size:14px;"> ● {action}</span>
      </div>

      <div style="background:#111e11; padding:8px 10px; border-radius:4px;
                  border-left:3px solid {color}; margin-bottom:6px;
                  font-size:11px; color:#c8e6c9;">
        {reas_html}
      </div>

      <div style="background:#1e1e11; padding:6px 10px; border-radius:4px;
                  border-left:3px solid #ffcc80; font-size:11px;">
        <b style="color:#ffcc80;">Justification:</b>
        <span style="color:#fff9c4;"> {justification}</span>
      </div>
    </div>"""


def render_history_detail(s):
    if not s["multi_turn"]:
        return "<div style='color:#aaa; padding:20px;'>Single-turn sample — no history.</div>"

    d = s["_debug"]
    html = f"""
    <div style="font-family:monospace; font-size:12px; background:#1e1e1e;
                color:#e0e0e0; padding:14px; border-radius:8px;">
      <b style="color:#90caf9;">Multi-turn State for ft_id: {s['ft_id']}</b><br>
      <b style="color:#aaa;">Dialogue: {str(s['dialogue_id'])[:50]}...</b>
      <b style="color:#aaa;"> | Current turn: {s['turn_id']}</b>
      <hr style="border:0.5px solid #333; margin:8px 0;">

      <b style="color:#90caf9;">Variable State at This Turn:</b><br>
      <span style="color:#00e676;">All known (cumulative): {d['all_known_this_turn']}</span><br>
      <span style="color:#ffcc80;">Cumulative known: {d['cumulative_known']}</span><br>
      <span style="color:#ff5252;">Still missing: {d['effective_missing']}</span><br>
      <span style="color:#aaa;">History-derived still missing: {d['still_missing_from_history']}</span>
      <hr style="border:0.5px solid #333; margin:8px 0;">

      <b style="color:#90caf9;">Full User Turn (what model sees):</b>
      <div style="background:#0d1117; padding:10px; border-radius:4px;
                  margin-top:4px; font-size:10px; white-space:pre-wrap;
                  max-height:300px; overflow-y:auto; color:#c9d1d9;">
{s['messages'][1]['content']}
      </div>

      <b style="color:#90caf9; margin-top:10px; display:block;">
        Full Assistant Turn (supervision signal):</b>
      <div style="background:#0d2010; padding:10px; border-radius:4px;
                  margin-top:4px; font-size:10px; white-space:pre-wrap;
                  max-height:300px; overflow-y:auto; color:#c8e6c9;">
{s['messages'][2]['content']}
      </div>
    </div>"""
    return html


def render_mistral_prompt(s):
    sys_t  = s["messages"][0]["content"]
    user_t = s["messages"][1]["content"]
    asst_t = s["messages"][2]["content"]
    prompt = f"[INST] {sys_t}\n\n{user_t} [/INST] {asst_t}</s>"
    esc    = (prompt.replace("&","&amp;")
                    .replace("<","&lt;")
                    .replace(">","&gt;")
                    .replace("\n","<br>"))
    char_count = len(prompt)
    token_est  = len(prompt.split())
    return f"""
    <div style="margin-bottom:6px; color:#aaa; font-size:11px;">
      ~{char_count} chars | ~{token_est} tokens (rough estimate)
    </div>
    <div style="font-family:monospace; font-size:11px; background:#0d1117;
                color:#c9d1d9; padding:14px; border-radius:8px;
                border:1px solid #30363d; white-space:pre-wrap;
                max-height:600px; overflow-y:auto;">
      {esc}
    </div>"""


def update(change=None):
    pool = get_pool(split_dd.value, action_dd.value,
                    source_dd.value, mt_dd.value, kg_dd.value)
    if not pool:
        with out:
            clear_output(wait=True)
            print("No samples match current filters.")
        return

    idx_slider.max   = len(pool) - 1
    idx_slider.value = min(idx_slider.value, idx_slider.max)
    s = pool[idx_slider.value]

    with out:
        clear_output(wait=True)
        print(f"Showing {idx_slider.value+1} of {len(pool)} | "
              f"{split_dd.value.upper()} | "
              f"action={action_dd.value} | source={source_dd.value} | "
              f"KG={kg_dd.value}")

        if view_dd.value == "Formatted":
            display(HTML(render_formatted(s)))
        elif view_dd.value == "Raw JSON":
            print(json.dumps(s, indent=2))
        elif view_dd.value == "Mistral Prompt":
            display(HTML(render_mistral_prompt(s)))
        elif view_dd.value == "History Detail":
            display(HTML(render_history_detail(s)))


for w in [split_dd, action_dd, source_dd,
          mt_dd, kg_dd, view_dd, idx_slider]:
    w.observe(update, names="value")

display(widgets.VBox([
    widgets.HTML(
        "<b style='color:#90caf9; font-size:14px;'>"
        "🔬 FT Dataset Viewer v2</b>"
    ),
    widgets.HBox([split_dd, action_dd, source_dd, mt_dd, kg_dd]),
    view_dd,
    idx_slider,
    out
]))

update()

In [ ]:
# reload all splits into all_ft
splits = {}
for name in ["train", "val", "test"]:
    path = f"{FT_OUT_DIR}/ft_{name}_debug.json"
    with open(path) as f:
        splits[name] = json.load(f)

all_ft = splits["train"] + splits["val"] + splits["test"]
print(f"Reloaded {len(all_ft)} samples")

In [ ]:
# check ABSTAIN generic
abstain_generic = [
    s for s in all_ft
    if s["action"] == "ABSTAIN"
    and any(
        not is_specific_missing(m)
        for m in s["_debug"]["effective_missing"]
    )
]
print(f"{'✅' if not abstain_generic else '⚠️'}  "
      f"ABSTAIN generic missing: {len(abstain_generic)}")

# check ASK clarification anchors to known variable
ask_bad_anchor = [
    s for s in all_ft
    if s["action"] == "ASK"
    and "<clarification_question>" in s["messages"][2]["content"]
    and not any(
        k.lower() in s["messages"][2]["content"].lower()
        for k in s["_debug"]["all_known_this_turn"]
        if len(k) > 3
    )
]
print(f"{'✅' if not ask_bad_anchor else '⚠️'}  "
      f"ASK bad anchor: {len(ask_bad_anchor)}")

# sample 5 clarification questions to eyeball
import random
ask_samples = [s for s in all_ft if s["action"] == "ASK"
               and "<clarification_question>" in s["messages"][2]["content"]]
for s in random.sample(ask_samples, min(5, len(ask_samples))):
    start = s["messages"][2]["content"].find("<clarification_question>") \
            + len("<clarification_question>")
    end   = s["messages"][2]["content"].find("</clarification_question>")
    print(f"  Q: {s['_debug']['query'][:60]}")
    print(f"  Known: {s['_debug']['all_known_this_turn'][:3]}")
    print(f"  Clarification: {s['messages'][2]['content'][start:end].strip()}")
    print()

In [ ]:
# check no ANSWER sample has ?unknown in graph
bad_answer = [s for s in ft_samples
              if s["action"] == "ANSWER"
              and any("?unknown" in t for t in s["_debug"]["graph_triples"])]
print(f"ANSWER samples with ?unknown: {len(bad_answer)}  "
      f"{'✅' if len(bad_answer)==0 else '❌'}")

# check ANSWER samples have known entity in graph
no_entity = [s for s in ft_samples
             if s["action"] == "ANSWER"
             and s["num_triples"] > 0
             and not any(
                 any(k.lower() in t.lower()
                     for k in s["_debug"]["all_known_this_turn"])
                 for t in s["_debug"]["graph_triples"]
             )]
print(f"ANSWER samples with irrelevant graph: {len(no_entity)}  "
      f"{'✅' if len(no_entity)==0 else '❌'}")

# skipped reason breakdown
print(f"\nSkipped: {dict(skipped)}")
print(f"Action dist: {dict(Counter(s['action'] for s in ft_samples))}")

In [ ]:
import json
import random
from collections import Counter, defaultdict
import numpy as np

random.seed(42)

# ── load all splits ───────────────────────────────────────────
splits = {}
for name in ["train", "val", "test"]:
    path = f"{FT_OUT_DIR}/ft_{name}_debug.json"
    with open(path) as f:
        splits[name] = json.load(f)

all_ft = splits["train"] + splits["val"] + splits["test"]
print(f"Total samples: {len(all_ft)}")


# ═══════════════════════════════════════════════════════════════
# SECTION 1 — GLOBAL STATS
# ═══════════════════════════════════════════════════════════════
def global_stats(samples, label="ALL"):
    print(f"\n{'='*60}")
    print(f"  {label}  (n={len(samples)})")
    print(f"{'='*60}")

    acts  = Counter(s["action"]  for s in samples)
    srcs  = Counter(s["source"]  for s in samples)
    diffs = Counter(s["difficulty"] for s in samples)
    fms   = Counter(s["failure_mode"] for s in samples)
    mt    = sum(1 for s in samples if s["multi_turn"])

    trip  = [s["num_triples"]      for s in samples]
    kn    = [s["num_known"]        for s in samples]
    ms    = [s["num_missing"]      for s in samples]
    nd    = [s["num_matched_nodes"] for s in samples]

    zero_t = sum(1 for t in trip if t == 0)
    zero_n = sum(1 for n in nd   if n == 0)

    print(f"\n  Actions:")
    for a in ["ANSWER","ASK","ABSTAIN"]:
        n = acts[a]
        print(f"    {a:10}: {n:6}  ({100*n/max(len(samples),1):.1f}%)")

    print(f"\n  Sources:")
    for src, cnt in srcs.most_common():
        print(f"    {src:15}: {cnt:6}  ({100*cnt/len(samples):.1f}%)")

    print(f"\n  Difficulty: {dict(diffs)}")
    print(f"  Failure modes: {dict(fms.most_common(5))}")
    print(f"  Multi-turn: {mt} ({100*mt/len(samples):.1f}%)")

    print(f"\n  Graph coverage:")
    print(f"    Avg triples      : {np.mean(trip):.2f}")
    print(f"    Avg matched nodes: {np.mean(nd):.2f}")
    print(f"    0-triple samples : {zero_t} ({100*zero_t/len(samples):.1f}%)")
    print(f"    0-node samples   : {zero_n} ({100*zero_n/len(samples):.1f}%)")

    print(f"\n  Variable state:")
    print(f"    Avg known   : {np.mean(kn):.2f}")
    print(f"    Avg missing : {np.mean(ms):.2f}")

    # token length estimate
    u_lens = [len(s["messages"][1]["content"].split()) for s in samples]
    a_lens = [len(s["messages"][2]["content"].split()) for s in samples]
    print(f"\n  Prompt lengths (words):")
    print(f"    User turn  — mean={np.mean(u_lens):.0f}  "
          f"max={max(u_lens)}  min={min(u_lens)}")
    print(f"    Asst turn  — mean={np.mean(a_lens):.0f}  "
          f"max={max(a_lens)}  min={min(a_lens)}")


global_stats(all_ft, "FULL DATASET")
for split_name, split_samples in splits.items():
    global_stats(split_samples, split_name.upper())


# ═══════════════════════════════════════════════════════════════
# SECTION 2 — RAW JSON SAMPLER
# stratified: action × source × turn type × edge cases
# ═══════════════════════════════════════════════════════════════
def get_samples(samples, action=None, source=None,
                multi_turn=None, has_graph=None,
                failure_mode=None, min_known=None,
                min_turn=None, n=3, seed=42):
    random.seed(seed)
    pool = samples
    if action:       pool = [s for s in pool if s["action"] == action]
    if source:       pool = [s for s in pool if s["source"] == source]
    if failure_mode: pool = [s for s in pool if s["failure_mode"] == failure_mode]
    if multi_turn is not None:
        pool = [s for s in pool if s["multi_turn"] == multi_turn]
    if has_graph is True:
        pool = [s for s in pool if s["num_triples"] > 0]
    if has_graph is False:
        pool = [s for s in pool if s["num_triples"] == 0]
    if min_known:
        pool = [s for s in pool if s["num_known"] >= min_known]
    if min_turn:
        pool = [s for s in pool
                if (s["turn_id"] or 0) >= min_turn]
    return random.sample(pool, min(n, len(pool)))


def print_sample(s, label="", show_full_messages=True):
    d = s["_debug"]
    print(f"\n{'─'*70}")
    print(f"  [{label}]  ft_id={s['ft_id']}  "
          f"src={s['source']}  action={s['action']}  "
          f"diff={s['difficulty']}  FM={s['failure_mode']}")
    print(f"  multi_turn={s['multi_turn']}  turn_id={s['turn_id']}  "
          f"triples={s['num_triples']}  nodes={s['num_matched_nodes']}")
    print(f"{'─'*70}")
    print(f"  QUERY        : {d['query']}")
    print(f"  ALL KNOWN    : {d['all_known_this_turn']}")
    print(f"  MISSING      : {d['missing_variables']}")
    print(f"  EFF MISSING  : {d['effective_missing']}")
    print(f"  GRAPH TRIPLES:")
    for t in d['graph_triples']:
        print(f"    {t}")
    print(f"  MATCHED NODES: {d['matched_nodes'][:3]}")
    print(f"  ORIG RESPONSE: {d['original_response'][:100]}")
    print(f"  HISTORY LEN  : {d['history_length']}")
    if d.get('resolved_vars'):
        print(f"  RESOLVED VARS: {d['resolved_vars']}")
    if show_full_messages:
        print(f"\n  ── USER TURN (what model sees) ──")
        print(s['messages'][1]['content'])
        print(f"\n  ── ASSISTANT TURN (supervision signal) ──")
        print(s['messages'][2]['content'])
    print(f"{'─'*70}")


# ═══════════════════════════════════════════════════════════════
# SECTION 3 — SYSTEMATIC CASE COVERAGE
# ═══════════════════════════════════════════════════════════════

CASES = [
    # label, filters
    ("ANSWER · single · hotpot · with graph",
     dict(action="ANSWER", source="hotpotqa",
          multi_turn=False, has_graph=True, n=2)),

    ("ANSWER · single · quac · with graph",
     dict(action="ANSWER", source="quac",
          multi_turn=False, has_graph=True, n=2)),

    ("ANSWER · single · no graph  ← EDGE CASE",
     dict(action="ANSWER", has_graph=False,
          multi_turn=False, n=2)),

    ("ASK · single · quac · with graph",
     dict(action="ASK", source="quac",
          multi_turn=False, has_graph=True, n=2)),

    ("ASK · single · sharc · with graph",
     dict(action="ASK", source="sharc",
          multi_turn=False, has_graph=True, n=2)),

    ("ASK · single · no graph  ← EDGE CASE",
     dict(action="ASK", has_graph=False,
          multi_turn=False, n=2)),

    ("ABSTAIN · single · quac",
     dict(action="ABSTAIN", source="quac",
          multi_turn=False, n=2)),

    ("ABSTAIN · single · contract_nli",
     dict(action="ABSTAIN", source="contract_nli",
          multi_turn=False, n=2)),

    ("ABSTAIN · single · sharc",
     dict(action="ABSTAIN", source="sharc",
          multi_turn=False, n=2)),

    ("ANSWER · multi-turn · early turn (turn 1-2)",
     dict(action="ANSWER", multi_turn=True,
          has_graph=True, n=2)),

    ("ASK · multi-turn · late turn (turn ≥5)  ← KEY CASE",
     dict(action="ASK", multi_turn=True,
          min_turn=5, has_graph=True, n=2)),

    ("ABSTAIN · multi-turn · many known vars  ← KEY CASE",
     dict(action="ABSTAIN", multi_turn=True,
          min_known=8, n=2)),

    ("ASK · multi-turn · high resolved vars  ← EDGE CASE",
     dict(action="ASK", multi_turn=True,
          min_turn=3, has_graph=True,
          failure_mode="INSUFFICIENT_VARIABLES", n=2)),

    ("ANSWER · contract_nli · COMPLETE",
     dict(action="ANSWER", source="contract_nli",
          failure_mode="COMPLETE", n=2)),

    ("ABSTAIN · contract_nli · INSUFFICIENT",
     dict(action="ABSTAIN", source="contract_nli",
          failure_mode="INSUFFICIENT_VARIABLES", n=2)),
]

print("\n\n" + "="*70)
print("  SECTION 3 — STRATIFIED SAMPLE INSPECTION")
print("="*70)

for label, filters in CASES:
    samples_pool = get_samples(all_ft, **filters)
    print(f"\n\n{'#'*70}")
    print(f"  CASE: {label}  ({len(samples_pool)} samples)")
    print(f"{'#'*70}")
    for s in samples_pool:
        print_sample(s, label=label, show_full_messages=True)


# ═══════════════════════════════════════════════════════════════
# SECTION 4 — WHAT THE MODEL SEES DURING FINETUNING
# ═══════════════════════════════════════════════════════════════
print("\n\n" + "="*70)
print("  SECTION 4 — FINETUNING INPUT FORMAT")
print("="*70)

sample = get_samples(all_ft, action="ASK",
                     multi_turn=True, has_graph=True, n=1)[0]

print("""
During finetuning the model receives the conversation as:

  messages = [
    {"role": "system",    "content": SYSTEM_PROMPT},
    {"role": "user",      "content": <user_turn>},
    {"role": "assistant", "content": <assistant_turn>}   ← THIS IS THE LABEL
  ]

The model is trained to predict the assistant turn given system+user.
Loss is computed ONLY on the assistant tokens.

For Mistral/LLaMA the tokenizer converts this to:
  <s>[INST] {system}\\n\\n{user} [/INST] {assistant}</s>

The assistant turn always has this structure:
  <reasoning>
    Step 1 — Query subject: ...
    Step 2 — Graph search: ...
    Step 3 — Variable check: ...
    Step 4 — Decision rationale: ...
  </reasoning>
  <decision>ANSWER|ASK|ABSTAIN</decision>
  <justification>one sentence</justification>
  [<clarification_question>...</clarification_question>]  ← ASK only

The model learns:
  1. To parse the graph context
  2. To reason step by step
  3. To output the correct action tag
  4. For ASK: to generate a specific clarification question
""")

print("── CONCRETE EXAMPLE (Mistral format) ──\n")
sys_t  = sample["messages"][0]["content"]
user_t = sample["messages"][1]["content"]
asst_t = sample["messages"][2]["content"]
prompt = f"[INST] {sys_t}\n\n{user_t} [/INST] {asst_t}</s>"
print(prompt[:3000])
print("\n... (truncated)")
print(f"\nTotal chars : {len(prompt)}")
print(f"Approx tokens: ~{len(prompt)//4}")


# ═══════════════════════════════════════════════════════════════
# SECTION 5 — QUALITY FLAGS
# ═══════════════════════════════════════════════════════════════
print("\n\n" + "="*70)
print("  SECTION 5 — QUALITY FLAGS")
print("="*70)

flags = {
    "ANSWER with ?unknown in graph"  : [s for s in all_ft
        if s["action"]=="ANSWER"
        and any("?unknown" in t for t in s["_debug"]["graph_triples"])],

    "ANSWER with 0 triples"          : [s for s in all_ft
        if s["action"]=="ANSWER" and s["num_triples"]==0],

    "ASK with no effective_missing"  : [s for s in all_ft
        if s["action"]=="ASK"
        and not s["_debug"]["effective_missing"]],

    "ABSTAIN with generic missing"   : [s for s in all_ft
        if s["action"]=="ABSTAIN"
        and any("eligibility" in m
                for m in s["_debug"]["effective_missing"])],

    "Prompt > 1500 tokens (est)"     : [s for s in all_ft
        if len(s["messages"][1]["content"].split()) > 1200],

    "ANSWER with irrelevant graph"   : [s for s in all_ft
        if s["action"]=="ANSWER"
        and s["num_triples"] > 0
        and not any(
            any(k.lower() in t.lower()
                for k in s["_debug"]["all_known_this_turn"])
            for t in s["_debug"]["graph_triples"]
        )],
}

all_clear = True
for flag_name, flagged in flags.items():
    status = "✅" if len(flagged)==0 else "⚠️"
    if len(flagged) > 0: all_clear = False
    print(f"  {status}  {flag_name:45}: {len(flagged)}")
    if flagged:
        # show one example
        print_sample(flagged[0],
                     label=f"EXAMPLE of: {flag_name}",
                     show_full_messages=False)

print(f"\n  {'✅ ALL CLEAR — ready for finetuning' if all_clear else '⚠️  Issues found — review flagged samples above'}")


# ═══════════════════════════════════════════════════════════════
# SECTION 6 — SAVE READABLE SAMPLES FOR MANUAL REVIEW
# ═══════════════════════════════════════════════════════════════
review = {}
for label, filters in CASES:
    pool = get_samples(all_ft, **filters)
    review[label] = pool
var_embs
with open(f"{FT_OUT_DIR}/manual_review_samples.json","w") as f:
    json.dump(review, f, indent=2)
print(f"\nSaved manual review → {FT_OUT_DIR}/manual_review_samples.json")